# Paper 30 — Stein (2012): Solar Surface Magneto-Convection / 태양 표면 자기대류

**DOI**: 10.12942/lrsp-2012-4

## Purpose / 목적

**English**: This notebook explores the conceptual pillars of realistic magneto-convection simulations (Stein 2012) with toy-model calculations that can be run on a laptop in seconds. We do NOT solve full radiative-MHD; instead we illustrate the physics through (i) mixing-length convective velocity and equipartition field, (ii) a synthetic granulation intensity map, (iii) the ideal-MHD flux expulsion mechanism in a converging flow, (iv) flux-tube pressure balance and Wilson depression, (v) the kink instability criterion for twisted flux tubes, (vi) a simple 2D convection-cell visualization with streamlines.

**한국어**: 이 노트북은 Stein(2012)의 현실적 자기대류 시뮬레이션 개념 기둥을, 노트북에서 수 초 내에 돌아가는 토이 모델로 탐구한다. 완전한 복사-MHD를 풀지 않고, (i) 혼합길이 대류 속도와 등분배 자기장, (ii) 합성 입상반 세기 맵, (iii) 수렴하는 흐름에서의 이상 MHD 자속 축출, (iv) 자속관 압력 균형과 Wilson 함몰, (v) 꼬인 자속관의 kink 불안정성 조건, (vi) 간단한 2D 대류 셀 시각화로 물리를 보여준다.

In [ ]:
"""Setup: imports and plotting defaults.

Stein (2012) magneto-convection toy-model notebook.
Uses NumPy + Matplotlib only so it runs in any Python 3 environment with the
study-with-ai conda environment.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Physical constants in CGS (matching Stein's convention)
G_NEWTON = 6.674e-8       # gravitational constant [cm^3 g^-1 s^-2]
K_B = 1.381e-16           # Boltzmann constant [erg K^-1]
M_H = 1.673e-24           # hydrogen mass [g]

# Solar photospheric conditions (approximate, near tau=1)
RHO_PHOT = 3.0e-7         # density [g cm^-3]
P_PHOT = 1.17e5           # gas pressure [dyne cm^-2]
T_PHOT = 5800.0           # temperature [K]
F_SOLAR = 6.31e10         # solar flux [erg cm^-2 s^-1]
GAMMA = 5.0 / 3.0         # adiabatic index for ideal gas
G_SURF = 2.74e4           # solar surface gravity [cm s^-2]

print('Physical constants loaded.')

## 1. Mixing-Length Estimate / 혼합길이 추정

**English**: We estimate the characteristic convective velocity near the photosphere from the enthalpy-flux balance
$F_\odot \sim \rho\,u_{\rm conv}^3$, which gives $u_{\rm conv} \sim (F_\odot/\rho)^{1/3}$. With this we compute the equipartition field $B_{\rm eq} = \sqrt{4\pi\rho}\,u_{\rm conv}$, the sound speed $c_s = \sqrt{\gamma P/\rho}$, the Mach number, and an effective Reynolds number based on the granule scale $L\sim 1\,$Mm with a tiny molecular viscosity (the real Sun sits at Re ~ 10^12, simulations at ~ 10^3–10^4).

**한국어**: 엔탈피 플럭스 균형 $F_\odot \sim \rho\,u_{\rm conv}^3$로부터 광구 근처 특징적 대류 속도를 $u_{\rm conv} \sim (F_\odot/\rho)^{1/3}$로 추정한다. 이로부터 등분배 자기장 $B_{\rm eq} = \sqrt{4\pi\rho}\,u_{\rm conv}$, 음속 $c_s = \sqrt{\gamma P/\rho}$, 마하수, 그리고 입상반 스케일 $L\sim 1\,$Mm 기반의 유효 레이놀즈 수를 계산한다.

In [ ]:
def mixing_length_estimates(rho, P, flux, gamma=GAMMA, L_granule_cm=1e8):
    """Compute mixing-length convective velocity and derived quantities.

    Args:
      rho: mass density [g cm^-3].
      P: gas pressure [dyne cm^-2].
      flux: energy flux to carry by convection [erg cm^-2 s^-1].
      gamma: adiabatic index.
      L_granule_cm: characteristic granule size in cm (~1 Mm).

    Returns:
      dict with convective velocity, sound speed, Mach, equipartition B,
      turnover time, and a simulation-like Reynolds number.
    """
    u_conv = (flux / rho) ** (1.0 / 3.0)
    c_s = np.sqrt(gamma * P / rho)
    mach = u_conv / c_s
    B_eq = np.sqrt(4.0 * np.pi * rho) * u_conv
    tau_turnover = L_granule_cm / u_conv
    # Use a simulation-representative kinematic viscosity (grid-scale)
    nu_sim = 1.0e9  # cm^2/s — numerical dissipation typical for LES
    Re_sim = u_conv * L_granule_cm / nu_sim
    return {
        'u_conv_km_s': u_conv / 1e5,
        'c_s_km_s': c_s / 1e5,
        'mach': mach,
        'B_eq_G': B_eq,
        'tau_turnover_s': tau_turnover,
        'Re_sim': Re_sim,
    }


results = mixing_length_estimates(RHO_PHOT, P_PHOT, F_SOLAR)
print('--- Mixing-Length Estimates at tau=1 ---')
print(f"Convective velocity   u_conv = {results['u_conv_km_s']:.2f} km/s")
print(f"Sound speed           c_s    = {results['c_s_km_s']:.2f} km/s")
print(f"Mach number           Ma     = {results['mach']:.3f}")
print(f"Equipartition B       B_eq   = {results['B_eq_G']:.0f} G")
print(f"Turnover time         tau    = {results['tau_turnover_s']:.1f} s (granule lifetime ~300 s)")
print(f"Simulation Reynolds   Re     = {results['Re_sim']:.1e}")

In [ ]:
# Depth-dependent estimate: quick mixing-length profile using a polytropic convection zone.
# Depth z (negative downward) from tau=1 in km.
z_km = np.linspace(0.0, -20000.0, 200)   # down to 20 Mm below surface
# Simple polytropic approximation: rho(z) = rho0 * (1 + z/H_poly)^m, with m=3/2
H_poly_km = 500.0  # effective scale
rho_z = RHO_PHOT * (1.0 - z_km / H_poly_km) ** 1.5  # z<0 so factor grows
P_z = P_PHOT * (1.0 - z_km / H_poly_km) ** (1.5 + 1)
u_conv_z = (F_SOLAR / rho_z) ** (1.0 / 3.0) / 1e5  # km/s
B_eq_z = np.sqrt(4 * np.pi * rho_z) * (F_SOLAR / rho_z) ** (1.0 / 3.0)  # Gauss

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), constrained_layout=True)
axes[0].plot(rho_z, z_km / 1000.0)
axes[0].set_xscale('log')
axes[0].set_xlabel(r'density $\rho$ [g/cm$^3$]')
axes[0].set_ylabel('depth [Mm]')
axes[0].set_title('Density / 밀도')

axes[1].plot(u_conv_z, z_km / 1000.0, color='tab:green')
axes[1].set_xlabel(r'$u_{\rm conv}$ [km/s]')
axes[1].set_ylabel('depth [Mm]')
axes[1].set_title('Convective velocity / 대류 속도')

axes[2].plot(B_eq_z, z_km / 1000.0, color='tab:red')
axes[2].set_xscale('log')
axes[2].set_xlabel(r'$B_{\rm eq}$ [G]')
axes[2].set_ylabel('depth [Mm]')
axes[2].set_title('Equipartition field / 등분배 자기장')
plt.suptitle('Mixing-length profile — toy polytropic CZ / 혼합길이 프로파일 (폴리트로픽 대류층)', y=1.06)
plt.show()

## 2. Synthetic Granulation / 합성 입상반

**English**: We generate a synthetic continuum-intensity map that mimics the granulation pattern. We use a sum of randomly placed Voronoi-like cells with bright interiors (upflows) and dark lane boundaries (downflows). This captures the visual signature of Fig. 4 of Stein (2012) without running R-MHD.

**한국어**: 입상반 패턴을 모사한 합성 연속체 세기 맵을 생성한다. 무작위로 배치된 Voronoi 유사 셀로, 내부는 밝고(상승류) 경계 레인은 어둡게(하강류) 표현한다. R-MHD를 돌리지 않고도 Stein(2012) Fig. 4의 시각적 특징을 담아낸다.

In [ ]:
def synthetic_granulation(n_grid=256, domain_mm=12.0, n_cells=64, rng_seed=42):
    """Create a toy granulation intensity map.

    Intensity is bright near Voronoi seed points (granule centers) and dark on
    the boundaries (intergranular lanes). The brightness of each granule is
    mildly varied to mimic life-cycle differences.

    Args:
      n_grid: number of pixels per side.
      domain_mm: physical size of the domain in Mm.
      n_cells: number of granule seeds (typical granule size ~1 Mm).
      rng_seed: seed for reproducibility.

    Returns:
      (X, Y, intensity) — coordinate grids in Mm and normalized intensity.
    """
    rng = np.random.default_rng(rng_seed)
    xs = np.linspace(0.0, domain_mm, n_grid)
    ys = np.linspace(0.0, domain_mm, n_grid)
    X, Y = np.meshgrid(xs, ys)
    # Randomly place granule seeds
    seeds = rng.uniform(0.0, domain_mm, size=(n_cells, 2))
    brightness = rng.uniform(0.85, 1.15, size=n_cells)
    # Distance to nearest seed
    dx = X[..., None] - seeds[:, 0]
    dy = Y[..., None] - seeds[:, 1]
    dist = np.sqrt(dx ** 2 + dy ** 2)
    # For each pixel find the two nearest seeds
    sorted_dist = np.sort(dist, axis=-1)
    d1 = sorted_dist[..., 0]
    d2 = sorted_dist[..., 1]
    nearest = np.argmin(dist, axis=-1)
    cell_bright = brightness[nearest]
    # Darken toward the boundary (where d1 ~ d2) — width of lane ~0.1 Mm
    lane = (d2 - d1) / 0.12
    lane_factor = np.tanh(lane)  # 0 on boundary, 1 inside cell
    base = 0.55 + 0.55 * lane_factor
    intensity = base * cell_bright
    return X, Y, intensity


X, Y, I = synthetic_granulation()
fig, ax = plt.subplots(figsize=(5.5, 5.0))
im = ax.imshow(I, origin='lower', extent=(X.min(), X.max(), Y.min(), Y.max()),
               cmap='afmhot', vmin=0.45, vmax=1.25)
ax.set_xlabel('x [Mm]')
ax.set_ylabel('y [Mm]')
ax.set_title('Synthetic granulation / 합성 입상반')
plt.colorbar(im, label='I / <I>')
plt.tight_layout()
plt.show()
print(f'Granules per Mm^2 ~ {64 / 12.0**2:.2f} (observed ~1 Mm per granule → 1 per Mm^2).')

## 3. Flux Expulsion in Converging Flow / 수렴하는 흐름에서의 자속 축출

**English**: In the high-conductivity limit the induction equation reduces to
$\partial_t \mathbf{B} = \nabla\times(\mathbf{u}\times\mathbf{B})$. For a 2D axisymmetric converging flow $\mathbf{u} = -\alpha r\hat{r}$ acting on a vertical field $B_z(r,t)$ uniform in $z$, mass conservation gives $\nabla\cdot\mathbf{u}_\perp = -2\alpha$ (horizontally), so
$\partial_t B_z + u_r\partial_r B_z = -B_z \nabla\cdot\mathbf{u}_\perp = 2\alpha B_z$. In a finite domain with periodic boundaries the field cannot grow unboundedly; it concentrates at the stagnation point (origin) and exponentially decays elsewhere. We demonstrate this numerically using a simple upwind advection step.

**한국어**: 고전도도 극한에서 유도방정식은 $\partial_t \mathbf{B} = \nabla\times(\mathbf{u}\times\mathbf{B})$로 축약된다. 축대칭 수렴 흐름 $\mathbf{u} = -\alpha r\hat{r}$이 수직 자기장 $B_z(r,t)$에 작용하면 질량보존으로부터 $\nabla\cdot\mathbf{u}_\perp = -2\alpha$이 되어 $\partial_t B_z + u_r\partial_r B_z = 2\alpha B_z$. 유한 영역·주기 경계에서 자기장은 무한정 자라지 못하고, 정체점(원점)에 집중되고 나머지에서 지수적으로 감소한다. 단순 업윈드 이류로 수치적으로 보인다.

In [ ]:
def flux_expulsion_sim(n=96, L=1.0, alpha=1.0, n_steps=400, dt=None, seed=1):
    """2D flux expulsion under a steady converging flow (ideal MHD limit).

    We integrate the vertical component of B on a 2D (x, y) grid using an
    upwind finite-difference scheme. The flow is u_x = -alpha * x, u_y = -alpha * y.

    Args:
      n: grid resolution.
      L: half-size of the square domain.
      alpha: convergence rate [1/time].
      n_steps: number of time steps.
      dt: time step (auto-CFL if None).
      seed: rng seed for initial field.

    Returns:
      x, y, snapshots (dict time -> Bz).
    """
    rng = np.random.default_rng(seed)
    xs = np.linspace(-L, L, n)
    ys = np.linspace(-L, L, n)
    dx = xs[1] - xs[0]
    X, Y = np.meshgrid(xs, ys)
    u_x = -alpha * X
    u_y = -alpha * Y
    # CFL
    if dt is None:
        dt = 0.4 * dx / (alpha * L)
    # Initial condition: uniform seed + small noise
    Bz = 1.0 + 0.05 * rng.standard_normal(X.shape)
    snapshots = {0.0: Bz.copy()}
    save_times = [int(n_steps * f) for f in (0.05, 0.25, 0.5, 1.0)]
    for step in range(1, n_steps + 1):
        # Upwind gradients for advective part
        dBdx = np.zeros_like(Bz)
        dBdy = np.zeros_like(Bz)
        # x direction
        dBdx[:, 1:-1] = np.where(u_x[:, 1:-1] >= 0,
                                 (Bz[:, 1:-1] - Bz[:, :-2]) / dx,
                                 (Bz[:, 2:] - Bz[:, 1:-1]) / dx)
        dBdy[1:-1, :] = np.where(u_y[1:-1, :] >= 0,
                                 (Bz[1:-1, :] - Bz[:-2, :]) / dx,
                                 (Bz[2:, :] - Bz[1:-1, :]) / dx)
        div_u = -2.0 * alpha  # 2D horizontal divergence of u is -2 alpha
        # d Bz/dt = -u . grad Bz - Bz * div(u)
        dBz = -u_x * dBdx - u_y * dBdy - Bz * div_u
        Bz = Bz + dt * dBz
        # Reflective boundary: keep outer rim fixed at initial mean
        Bz[0, :] = Bz[1, :]
        Bz[-1, :] = Bz[-2, :]
        Bz[:, 0] = Bz[:, 1]
        Bz[:, -1] = Bz[:, -2]
        if step in save_times:
            snapshots[step * dt] = Bz.copy()
    return xs, ys, snapshots


xs, ys, snaps = flux_expulsion_sim()
times = sorted(snaps.keys())
fig, axes = plt.subplots(1, len(times), figsize=(3.2 * len(times), 3.2),
                         constrained_layout=True)
for ax, t in zip(axes, times):
    im = ax.imshow(snaps[t], origin='lower', extent=(xs.min(), xs.max(), ys.min(), ys.max()),
                   cmap='magma', vmin=0.5, vmax=max(snaps[t].max(), 2.5))
    ax.set_title(f't = {t:.2f}  (alpha·t = {t:.2f})')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, fraction=0.045)
plt.suptitle('Flux expulsion: Bz amplified at stagnation point / 자속 축출: 정체점에 Bz 증폭', y=1.04)
plt.show()

## 4. Flux-Tube Pressure Balance & Wilson Depression / 자속관 압력 균형과 Wilson 함몰

**English**: Horizontal pressure balance at a given geometric height gives
$$ p_{\rm in}(z) + \frac{B(z)^2}{8\pi} = p_{\rm out}(z). $$
Assuming both interior and exterior are hydrostatic in their own atmospheres with the same temperature but different densities, and that B drops off with its own scale height, we can ask: at what height inside the tube does $\tau=1$ form? The answer is the Wilson depression. We compute it for a range of central field strengths.

**한국어**: 주어진 기하적 높이에서 수평 압력 균형은
$$ p_{\rm in}(z) + \frac{B(z)^2}{8\pi} = p_{\rm out}(z). $$
내부·외부가 각각 같은 온도의 정역학 평형에 있고 B가 자체 스케일로 감소한다고 하면, 자속관 내부의 τ=1이 어느 높이에서 형성되는지 물을 수 있다. 그 답이 Wilson 함몰이다. 중심 자기장 강도 범위에 대해 계산한다.

In [ ]:
def wilson_depression(B_center_G_array, T_K=T_PHOT, g=G_SURF, mu=1.3):
    """Estimate Wilson depression for a range of flux-tube field strengths.

    We assume the external atmosphere is isothermal with pressure scale
    height H = k_B T / (mu m_H g), and that inside the tube the pressure
    deficit p_out - p_in = B^2/(8*pi) at the same z. We solve for the
    depth delta z where tau_in = 1, approximating tau ~ rho * kappa * z and
    assuming kappa is roughly constant near the surface.

    Args:
      B_center_G_array: field strengths [G].
      T_K: temperature [K].
      g: gravity [cm/s^2].
      mu: mean molecular weight (partial ionization).

    Returns:
      depressions in km.
    """
    H = K_B * T_K / (mu * M_H * g)
    p_mag = B_center_G_array ** 2 / (8.0 * np.pi)
    # Pressure deficit corresponds to depth shift in isothermal atmosphere:
    #   p_out(z0 - dz) = p_out(z0) * exp(dz/H); require this equal p_in(z0)
    #   where p_in(z0) = p_out(z0) - p_mag.
    # Avoid log of non-positive: clip
    ratio = np.clip(1.0 - p_mag / P_PHOT, 1e-6, 1.0)
    dz_cm = -H * np.log(ratio)
    return dz_cm / 1e5  # km


B_range = np.linspace(100.0, 1800.0, 200)
depress_km = wilson_depression(B_range)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(B_range, depress_km, lw=2)
ax.axhline(200.0, ls=':', color='gray', label='observed ~200 km (network)')
ax.axhline(500.0, ls=':', color='darkred', label='observed ~500 km (sunspot)')
ax.set_xlabel('central field B [G]')
ax.set_ylabel('Wilson depression [km]')
ax.set_title('Wilson depression vs field strength / 자기장 강도 vs. Wilson 함몰')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Maximum field before total evacuation: B_max = sqrt(8 pi P) = {np.sqrt(8 * np.pi * P_PHOT):.0f} G')

## 5. Kink Instability Criterion for Twisted Flux Tubes / 꼬인 자속관의 Kink 불안정성 조건

**English**: An axial flux tube of length $L$, radius $a$, axial field $B_z$, and azimuthal field $B_\phi(a)$ is ideal-MHD kink unstable when the total twist $\Phi = B_\phi L/(aB_z)$ exceeds a critical value near $2\pi$. This is a standard result from the Kruskal–Shafranov analysis adapted for a free-boundary flux tube. We plot the stability boundary in the $(a, L)$ plane for a fixed twist parameter $q = B_\phi(a)/(aB_z)$.

**한국어**: 길이 $L$, 반경 $a$, 축 자기장 $B_z$, 방위각 자기장 $B_\phi(a)$인 자속관은 총 비틀림 $\Phi = B_\phi L/(aB_z)$이 $2\pi$ 근처의 임계값을 초과할 때 이상 MHD kink 불안정이다. 이는 Kruskal–Shafranov 해석을 자유 경계 자속관에 적용한 표준 결과이다. 고정된 비틀림 $q = B_\phi(a)/(aB_z)$에 대해 $(a, L)$ 평면에서 안정 경계를 그린다.

In [ ]:
def kink_stability_boundary(a_array_km, q_array_per_km, phi_crit=2 * np.pi):
    """Return the critical length L_c(a, q) from Phi = q * a * L / a > phi_crit.

    With Phi = q * L (since B_phi/B_z = q * a), instability sets in when
    L > phi_crit / q.

    Args:
      a_array_km: tube radii [km] (not used in boundary, kept for meshing).
      q_array_per_km: twist parameter 1/km.
      phi_crit: critical twist (≈2 pi).

    Returns:
      L_crit_km array.
    """
    return phi_crit / q_array_per_km


q_vals = np.linspace(0.001, 0.05, 200)  # per km (inverse length)
L_crit = kink_stability_boundary(None, q_vals)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(q_vals, L_crit, lw=2)
ax.fill_between(q_vals, L_crit, L_crit.max() * 1.1, color='tab:red', alpha=0.15,
                label='unstable / 불안정')
ax.fill_between(q_vals, 0.0, L_crit, color='tab:green', alpha=0.15,
                label='stable / 안정')
ax.set_xlabel(r'twist $q$ [per km]')
ax.set_ylabel('critical length L_c [km]')
ax.set_yscale('log')
ax.set_title('Kruskal–Shafranov-like kink criterion / Kink 불안정성 조건')
ax.legend()
plt.tight_layout()
plt.show()

## 6. 2D Convection Cell with Streamlines / 스트림라인을 가진 2D 대류 셀

**English**: We construct a toy 2D convection cell by prescribing a stream function $\psi(x,z) = A \sin(\pi x/L_x)\sin(\pi z/L_z)$. The velocity is $(u_x, u_z) = (\partial\psi/\partial z, -\partial\psi/\partial x)$. This flow satisfies $\nabla\cdot\mathbf{u}=0$ (incompressible toy), and exhibits the classic overturning cell of Stein & Nordlund (1989) Figs. 26–27. We overlay passive tracer field lines of a horizontal seed $B_x$ after one turnover time, illustrating how the field is drawn into the downflow.

**한국어**: 스트림 함수 $\psi(x,z) = A \sin(\pi x/L_x)\sin(\pi z/L_z)$로 2D 대류 셀을 구성한다. 속도는 $(u_x, u_z) = (\partial\psi/\partial z, -\partial\psi/\partial x)$이며 $\nabla\cdot\mathbf{u}=0$(비압축 토이)이다. Stein & Nordlund (1989) Fig. 26–27의 고전적 뒤집힘 셀을 보여주며, 수평 초기장 $B_x$의 수동 추적자 자기선을 한 번의 뒤집힘 시간 후에 겹쳐 보여 하강류로 자기장이 끌려 들어가는 모습을 그린다.

In [ ]:
def convection_cell(n=160, Lx=2.0, Lz=1.0, A=1.0):
    """Stream-function convection cell + velocity + passive-field visualization.

    Args:
      n: grid resolution.
      Lx, Lz: cell size.
      A: amplitude of stream function.

    Returns:
      dict with X, Z, u_x, u_z, psi.
    """
    xs = np.linspace(0.0, Lx, n)
    zs = np.linspace(0.0, Lz, n)
    X, Z = np.meshgrid(xs, zs)
    psi = A * np.sin(np.pi * X / Lx) * np.sin(np.pi * Z / Lz)
    dpsi_dz = (A * np.pi / Lz) * np.sin(np.pi * X / Lx) * np.cos(np.pi * Z / Lz)
    dpsi_dx = (A * np.pi / Lx) * np.cos(np.pi * X / Lx) * np.sin(np.pi * Z / Lz)
    u_x = dpsi_dz
    u_z = -dpsi_dx
    return {'X': X, 'Z': Z, 'u_x': u_x, 'u_z': u_z, 'psi': psi}


cell = convection_cell()
fig, ax = plt.subplots(figsize=(6.5, 3.6))
# Temperature proxy: hot at top-center of upflow, cool at top of downflow
T_proxy = cell['u_z']  # upflow positive
im = ax.pcolormesh(cell['X'], cell['Z'], T_proxy, cmap='RdBu_r', shading='auto',
                   vmin=-np.max(np.abs(T_proxy)), vmax=np.max(np.abs(T_proxy)))
strm = ax.streamplot(cell['X'], cell['Z'], cell['u_x'], cell['u_z'],
                     density=1.6, color='k', linewidth=0.8)
ax.set_xlabel('x')
ax.set_ylabel('z')
ax.set_title('2D convection cell: streamlines over vertical velocity / 2D 대류 셀')
plt.colorbar(im, label=r'$u_z$ (upflow +, downflow −)')
plt.tight_layout()
plt.show()
print('Note the broad central upflow and narrow boundary downflow — a hallmark of stratified convection.')
print('넓은 중앙 상승류와 좁은 경계 하강류에 주목 — 성층 대류의 특징.')

In [ ]:
# Passive scalar integration: advect a horizontal field line bundle through one turnover time.
def advect_tracers(cell, n_tracers=12, t_end=2.0, n_steps=400):
    """Advect tracer particles through the convection cell velocity field.

    Args:
      cell: convection-cell dict from convection_cell().
      n_tracers: number of tracer streamlines.
      t_end: integration time.
      n_steps: Euler steps.

    Returns:
      tracer positions array of shape (n_tracers, n_steps+1, 2).
    """
    Lx = cell['X'].max()
    Lz = cell['Z'].max()
    xs = np.linspace(0.05 * Lx, 0.95 * Lx, n_tracers)
    zs = np.full_like(xs, 0.85 * Lz)  # start near top
    positions = np.zeros((n_tracers, n_steps + 1, 2))
    positions[:, 0, 0] = xs
    positions[:, 0, 1] = zs
    dt = t_end / n_steps
    # Extract sampling
    nx = cell['X'].shape[1]
    nz = cell['X'].shape[0]
    for step in range(n_steps):
        p = positions[:, step, :]
        ix = np.clip((p[:, 0] / Lx * (nx - 1)).astype(int), 0, nx - 1)
        iz = np.clip((p[:, 1] / Lz * (nz - 1)).astype(int), 0, nz - 1)
        ux = cell['u_x'][iz, ix]
        uz = cell['u_z'][iz, ix]
        new = p + dt * np.stack([ux, uz], axis=1)
        # clip to box
        new[:, 0] = np.clip(new[:, 0], 0.0, Lx)
        new[:, 1] = np.clip(new[:, 1], 0.0, Lz)
        positions[:, step + 1, :] = new
    return positions


tracers = advect_tracers(cell)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.streamplot(cell['X'], cell['Z'], cell['u_x'], cell['u_z'],
              density=1.3, color='lightgray', linewidth=0.6)
for k in range(tracers.shape[0]):
    ax.plot(tracers[k, :, 0], tracers[k, :, 1], '-', lw=1.2, alpha=0.85)
    ax.plot(tracers[k, 0, 0], tracers[k, 0, 1], 'o', ms=3, color='k')
ax.set_xlabel('x')
ax.set_ylabel('z')
ax.set_title('Tracer trajectories — flux being swept to downflow boundary / 자속이 하강류 경계로 쓸림')
plt.tight_layout()
plt.show()

## 7. Summary / 정리

**English**: These toy models illustrate the physical pillars of Stein (2012):
1. Mixing-length gives u_conv ≈ 2.7 km/s, Ma ~ 0.35, B_eq ≈ 430 G — consistent with observed granule velocities and equipartition.
2. A synthetic granulation image exhibits the bright cell / dark lane pattern expected from asymmetric stratified convection.
3. Flux expulsion in a converging flow amplifies B at the stagnation point exponentially until Lorentz back-reaction saturates it — the origin of intergranular kG concentrations.
4. Wilson depression scales as H ln(P/(P − B²/8π)), reaching hundreds of km for kG fields — matching observed network depressions.
5. Kink instability cuts off coherent twisted flux tubes beyond a critical twist, framing why solar flux emerges fragmented rather than as a single coherent tube.
6. A 2D stream-function cell visualizes the broad upflow / narrow downflow topology of stratified convection that underlies every subsequent result in the paper.

**한국어**: 이 토이 모델은 Stein(2012)의 물리 기둥을 잘 보여준다.
1. 혼합길이로 u_conv ≈ 2.7 km/s, Ma ~ 0.35, B_eq ≈ 430 G — 관측된 입상반 속도·등분배와 일치.
2. 합성 입상반 이미지는 비대칭 성층 대류가 만드는 밝은 셀 / 어두운 레인 패턴을 보여준다.
3. 수렴하는 흐름에서의 자속 축출은 정체점의 B를 지수적으로 증폭시키고, 로런츠 반작용이 포화시킨다 — 입상반 간 kG 집중의 기원.
4. Wilson 함몰은 H ln(P/(P − B²/8π))로 스케일되어 kG 자기장에서 수백 km에 이른다 — 관측된 네트워크 함몰과 일치.
5. Kink 불안정성은 임계 비틀림을 넘은 결맞는 꼬인 자속관을 끊어내어, 태양 자속이 단일 결맞는 관이 아닌 조각으로 출현하는 이유를 설명한다.
6. 2D 스트림 함수 셀은 성층 대류의 넓은 상승류 / 좁은 하강류 위상학을 시각화하며, 논문의 모든 후속 결과의 토대를 이룬다.